# Figure 1 | BASSFISH workflow and detection

Decoded transcript maps, single-gene density maps, and mHTT puncta overlays.

Check input/output paths and settings before running each workflow. Run its cells in order with a fresh kernel. See README.md for required inputs, panel coverage, and configuration differences.

| Panels | Workflow |
|---|---|
| A, example | Decoded ROI scatter |
| F | Whole-brain and enlarged transcript maps |
| G | Single-gene density maps |
| H, detection overlay | MATLAB spot detection and Python overlay |
| B-E and H, raw fluorescence | Images/schematics; the E decoding example script is absent |


## MATLAB preprocessing

Run the MATLAB code in the next two Markdown cells separately in MATLAB.

- `Figure_1_a_1.m` prepares `rois.csv` from decoded HDF5 tables.
- `spot_verified.m` filters the mHTT image and exports detected spots. It requires the external MATLAB functions `bpass` and `pkfnd`, which are not included.

The spot-detection example writes a headerless three-column CSV. The Python overlay expects columns named `x`, `y`, and `brightness`; prepare the input with these headers before running the overlay.


### Figure_1_a_1.m

```matlab
clc
clear

a = double(h5read(['hd240625_s94_c1_decode.h5'], '/dataset')) ;
b = double(h5read(['hd240625_s94_c2_decode.h5'], '/dataset')) ;
c = double(h5read(['hd240625_s94_c3_decode.h5'], '/dataset')) ;

raw_s = [a;b;c];

raw_s(raw_s(:,5)~=raw_s(:,10),:)=[];
rois = raw_s(raw_s(:,1)>83&raw_s(:,2)>644&raw_s(:,1)<440&raw_s(:,2)<996,:);
writematrix(rois,'rois.csv')

A=rois;

x = A(:,1);
y = A(:,2);
geneID = A(:,9);

figure;
scatter(x, y, 10, geneID, 'filled');
axis equal;
set(gca, 'YDir', 'reverse');

colormap(hsv(191));
caxis([1 191]);

cb = colorbar;
cb.Label.String = 'Gene ID (1–191)';
```


### spot_verified.m

```matlab
clc
clear

a = double(imread("C1-MAX_mhtt_dapi_Composite-2.tif"));

a = bpass(a,3,4);

b = pkfnd(a,200,3);
for i1 = 1:length(b)
    b(i1,3) = a(b(i1,2),b(i1,1));

end

figure

hold on
imshow(a,[100,500])
hold on
scatter(b(:,1),b(:,2))

csvwrite('spot.csv',b)
```


## Panel A | Decoded ROI scatter

Requires `rois.csv` in the working directory. Generate it with the MATLAB example or provide a compatible table. Points are colored by factorized gene identity. The manuscript legend describes total signals per cell; that coloring routine is not included.


In [ ]:
# Imports and settings.
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable


plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


df = pd.read_csv("rois.csv")


x = df.iloc[:, 0]
y = df.iloc[:, 1]
gene_col = df.iloc[:, 8]


gene_codes, gene_uniques = pd.factorize(gene_col)
n_gene = len(gene_uniques)
print(f"{n_gene} gene categories are used for coloring.")


cm = 1 / 2.54
fig, ax = plt.subplots(
    figsize=(6 * cm, 6 * cm),
    facecolor="black"
)
ax.set_facecolor("black")

sc = ax.scatter(
    x, y,
    c=gene_codes,
    s=0.2,
    marker="o",
    cmap="nipy_spectral"
)

ax.set_aspect("equal")
ax.invert_yaxis()


ax.set_xticks([])
ax.set_yticks([])


divider = make_axes_locatable(ax)

cax = divider.append_axes("right", size="5%", pad=0.05)

cbar = plt.colorbar(sc, cax=cax)
cbar.set_label("Gene index (factorized)", color="white")
cbar.ax.tick_params(color="white")
for t in cbar.ax.get_yticklabels():
    t.set_color("white")

plt.tight_layout()


plt.savefig("rois_genes_black_bg_4p8cm.png", dpi=600, bbox_inches="tight")
plt.savefig("rois_genes_black_bg_4p8cm.pdf", bbox_inches="tight")

plt.show()


gene_table = pd.DataFrame({
    "code": range(n_gene),
    "gene": gene_uniques
})
gene_table.to_csv("gene_code_map.csv", index=False)


## Panels F-G | Transcript scatter and density maps

Requires decoded Parquet files. The whole-brain gene list includes `Ccdc153` in addition to the 15 genes listed in the legend. Two cortex calls write `region_ctx.png`; the second overwrites the first. This workflow plots decoded signals; the panel E decoding example is not included.


In [ ]:
# Imports and settings.
import numpy as np
import pandas as pd
import os
import re
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.stats import kde
import scanpy as sc
import matplotlib as mpl


In [ ]:
# Imports and settings.
from pathlib import Path
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

def draw_region(
    markers=None,
    colors=None,
    xmin=0, xmax=84000,
    ymin=0, ymax=42000,
    data_dir=r"E:/CL/HD/script_jc/hd240625/output/filtered_dedup_deoverlap",
    out_png=r"E:/CL/HD/PAPER_WRITE/Figure1/figures/markers_str.png",
    dpi=300,
    ss=0.1
):
    """Plot decoded transcript coordinates on a black background."""


    mpl.rcParams['savefig.transparent'] = False
    mpl.rcParams['savefig.facecolor'] = 'black'
    mpl.rcParams['savefig.edgecolor'] = 'black'

    data_dir = Path(data_dir)
    out_png = Path(out_png)
    color_map = dict(zip(markers, colors))


    dfs = []
    for mk in markers:
        fp = data_dir / f"{mk}.final.parquet"
        if not fp.exists():
            print(f"[WARN] File not found: {fp}; skipping this marker")
            continue
        df_mk = pd.read_parquet(fp)
        df_mk["marker"] = mk
        dfs.append(df_mk)

    if not dfs:
        print("No marker data were loaded successfully.")
        return None

    df_all = pd.concat(dfs, ignore_index=True)

    df_all = df_all.query(f"{ymin} < global_y < {ymax} and {xmin} < global_x < {xmax}")
    if df_all.empty:
        print("No points in the selected region.")
        return df_all


    width  = int((df_all["global_x"].max() - df_all["global_x"].min()) / 3000)
    height = int((df_all["global_y"].max() - df_all["global_y"].min()) / 3000)
    width  = max(width, 2)
    height = max(height, 2)


    fig, ax = plt.subplots(figsize=(width, height))
    fig.patch.set_facecolor('black')
    fig.patch.set_alpha(1.0)
    ax.set_facecolor('black')
    ax.patch.set_alpha(1.0)

    for mk in markers:
        sub = df_all[df_all["marker"] == mk]
        if sub.empty:
            continue
        ax.scatter(
            sub["global_x"], sub["global_y"],
            s=ss, alpha=1.0, marker="o",
            edgecolors="none", linewidths=0,
            color=color_map.get(mk, "white"),
            label=mk
        )

    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)


    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

    fig.savefig(
        out_png,
        dpi=dpi,
        facecolor='black',
        edgecolor='none',
        transparent=False,
        bbox_inches=None,
        pad_inches=0
    )
    plt.show()
    plt.close(fig)

    print(f"Saved to: {out_png}")


In [ ]:
gene_colors = {

    'Sst':"#a30a9e",
    'Pvalb':"#5d49f1",
    'Vip':"#49f1ee",
    'Gja1':"#f49734",
    'Slc17a7': '#ff7f50',
    'Slc17a6': '#ffb6c1',
    'Gad1':    '#fff68f',
    'Lamp5':   '#ff4040',
    'Slc1a3':  '#008000',
    'Drd1':    '#2acaea',
    'Drd2':    '#0a75ad',
    'Calb2':   '#afeeee',
    'Cldn11':  '#00ff00',
    'Vtn':     '#cc0000',
    'Ccdc153': '#00ffff',
    'Cpne4':  '#ff00ff',
}

markers = list(gene_colors.keys())
colors  = [gene_colors[g] for g in markers]

draw_region(
    markers=markers,
    colors=colors,
    xmin=0, xmax=84000,
    ymin=0, ymax=42000,
    out_png=r"E:/CL/HD/PAPER_WRITE/Figure1/figures/region_full.png",
    dpi = 300,
    ss = 0.4
)


In [ ]:
gene_colors = {
    'Sst':"#a30a9e",

    'Vip':"#49f1ee",
    'Gja1':"#f49734",


    'Gad1':    '#fff68f',
    'Lamp5':   '#ff4040',
    'Slc1a3':  '#008000',
    'Drd1':    '#2acaea',
    'Drd2':    '#0a75ad',

    'Cldn11':  '#00ff00',
    'Vtn':     '#cc0000',
    'Ccdc153': '#00ffff',
    'Cpne4':  '#ff00ff',
}
markers = ['Mbp','Sst','Gja1','Drd1','Drd2']
markers = list(gene_colors.keys())
colors  = [gene_colors[g] for g in markers]

draw_region(
    markers=markers,
    colors=colors,
    xmin=31000, xmax=33000,
    ymin=21000, ymax=23000,
    out_png=r"E:/CL/HD/PAPER_WRITE/Figure1/figures/region_str.png",
    dpi = 300,
    ss = 1
)


In [ ]:
# Imports and settings.
from pathlib import Path
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt


def draw_region(
    markers=None,
    colors=None,
    sizes=None,
    xmin=0, xmax=84000,
    ymin=0, ymax=42000,
    data_dir=r"E:/CL/HD/script_jc/hd240625/output/filtered_dedup_deoverlap",
    out_png=r"E:/CL/HD/PAPER_WRITE/Figure1/figures/markers_str.png",
    dpi=300,
    ss=0.1
):
    """Plot decoded transcript coordinates on a black background."""


    mpl.rcParams['savefig.transparent'] = False
    mpl.rcParams['savefig.facecolor'] = 'black'
    mpl.rcParams['savefig.edgecolor'] = 'black'

    if markers is None or colors is None:
        raise ValueError("markers and colors must not be empty.")
    if len(markers) != len(colors):
        raise ValueError("markers and colors must have the same length.")

    data_dir = Path(data_dir)
    out_png = Path(out_png)
    color_map = dict(zip(markers, colors))


    if sizes is None:
        size_map = {mk: ss for mk in markers}
    elif isinstance(sizes, dict):
        size_map = {mk: float(sizes.get(mk, ss)) for mk in markers}
    else:
        if len(sizes) != len(markers):
            raise ValueError("If sizes is a list, its length must match markers.")
        size_map = {mk: float(s) for mk, s in zip(markers, sizes)}


    dfs = []
    for mk in markers:
        fp = data_dir / f"{mk}.final.parquet"
        if not fp.exists():
            print(f"[WARN] File not found: {fp}; skipping this marker")
            continue
        df_mk = pd.read_parquet(fp)
        df_mk["marker"] = mk
        dfs.append(df_mk)

    if not dfs:
        print("No marker data were loaded successfully.")
        return None

    df_all = pd.concat(dfs, ignore_index=True)


    df_all = df_all.query(f"{ymin} < global_y < {ymax} and {xmin} < global_x < {xmax}")
    if df_all.empty:
        print("No points in the selected region.")
        return df_all


    width  = int((df_all["global_x"].max() - df_all["global_x"].min()) / 3000)
    height = int((df_all["global_y"].max() - df_all["global_y"].min()) / 3000)
    width  = max(width, 2)
    height = max(height, 2)


    fig, ax = plt.subplots(figsize=(width, height))
    fig.patch.set_facecolor('black'); fig.patch.set_alpha(1.0)
    ax.set_facecolor('black'); ax.patch.set_alpha(1.0)


    for mk in markers:
        sub = df_all[df_all["marker"] == mk]
        if sub.empty:
            continue
        ax.scatter(
            sub["global_x"], sub["global_y"],
            s=size_map.get(mk, ss)*0.8,
            alpha=1.0, marker="o",
            edgecolors="none", linewidths=0,
            color=color_map.get(mk, "white"),
            label=mk
        )

    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

    fig.savefig(
        out_png,
        dpi=dpi,
        facecolor='black',
        edgecolor='none',
        transparent=False,
        bbox_inches=None,
        pad_inches=0
    )
    plt.show()
    plt.close(fig)

    print(f"Saved to: {out_png}")


gene_colors = {
     'Slc17a7': '#ff7f50',
     'Slc1a3':  '#008000',
     'Gja1':"#f49734",
     'Slc17a6': '#ffb6c1',
     'Gad1':    '#fff68f',
     'Drd1':    '#2acaea',
     'Drd2':    '#0a75ad',
     'Calb2':   '#afeeee',
     'Vip':"#49f1ee",
     'Vtn':     '#cc0000',
     'Ccdc153': '#00ffff',
     'Cpne4':  '#ff00ff',
     'Cldn11':  '#00ff00',
     'Lamp5':   "#ff4040",
     'Sst':"#a30a9e",
     'Pvalb':"#5d49f1",
}

markers = list(gene_colors.keys())
colors  = [gene_colors[g] for g in markers]


gene_sizes = {
    'Slc17a7': 0.3,
    'Slc17a6': 0.5,
    'Gad1':    0.5,
    'Drd1':    1,
    'Drd2':    1,
    'Slc1a3':  0.3,
    'Gja1':    1,
    'Pvalb':   1,
    'Vip':     1,
    'Sst':     1,
    'Lamp5':   1,
    'Calb2':   1,
    'Cldn11':  1,
    'Vtn':     1,
    'Ccdc153': 1,
    'Cpne4':   1,
}

k1 = 29500
k2 = 6800

draw_region(
    markers=markers,
    colors=colors,
    sizes=gene_sizes,
    xmin=k1, xmax=k1+2000,
    ymin=k2, ymax=k2+2000,
    out_png=r"E:/CL/HD/PAPER_WRITE/Figure1/figures/region_ctx.png",
    dpi=300,
    ss=1
)


In [ ]:
gene_colors = {
    'Gja1':"#f49734",
    'Slc17a6': '#ffb6c1',
    'Gad1':    '#fff68f',
    'Slc17a7': '#ff7f50',
    'Slc1a3':  '#008000',
    'Drd1':    '#2acaea',
    'Drd2':    '#0a75ad',
    'Calb2':   '#afeeee',
    'Vip':"#49f1ee",
    'Vtn':     '#cc0000',
    'Ccdc153': '#00ffff',
    'Cpne4':  '#ff00ff',
    'Cldn11':  '#00ff00',
    'Lamp5':   '#ff4040',
    'Sst':"#a30a9e",
    'Pvalb':"#5d49f1",
}

markers = list(gene_colors.keys())
colors  = [gene_colors[g] for g in markers]

k1 =30000
k2 =7000
draw_region(
    markers=markers,
    colors=colors,
    xmin=k1, xmax=k1+3000,
    ymin=k2, ymax=k2+3000,
    out_png=r"E:/CL/HD/PAPER_WRITE/Figure1/figures/region_ctx.png",
    dpi = 300,
    ss = 0.2
)


In [ ]:
# Imports and settings.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import copy

def draw_density_with_separate_cbar(
    marker,
    x_range=(0, 84000),
    y_range=(0, 42000),
    bins=300,
    cmap='viridis',

    dmin=None,
    dmax=None,
    main_png=None,
    cbar_png=None
):
    """Plot transcript density with a separate normalized colorbar."""


    file_tpl = Path(
        'E:/CL/HD/script_jc/hd240625/output/filtered_dedup_deoverlap/{marker}.final.parquet'
    )
    fp = file_tpl.with_name(f'{marker}.final.parquet')
    df = pd.read_parquet(fp)


    df = df.query(
        f'{y_range[0]} < global_y < {y_range[1]} and '
        f'{x_range[0]} < global_x < {x_range[1]}'
    )

    x = df['global_x'].to_numpy()
    y = df['global_y'].to_numpy()

    if x.size == 0:
        print(f'{marker}: no points in this range')
        return


    H, xedges, yedges = np.histogram2d(
        x, y,
        bins=bins,
        range=[x_range, y_range]
    )

    H = H.astype(float)


    mask_zero = (H == 0)
    H_nonzero = H[~mask_zero]

    if H_nonzero.size == 0:
        print(f'{marker}: all bins in this range are zero')
        return


    if dmin is None:
        dmin = float(H_nonzero.min())
    if dmax is None:
        dmax = float(H_nonzero.max())
    if dmax <= dmin:
        dmax = dmin + 1e-9


    H_norm = (H - dmin) / (dmax - dmin)

    H_norm = np.clip(H_norm, 0.0, 1.0)


    H_norm[mask_zero] = np.nan

    extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]


    cmap_obj = copy.copy(plt.get_cmap(cmap))
    cmap_obj.set_bad((1, 1, 1, 0))


    width  = int((x.max() - x.min()) / 3000)
    height = int((y.max() - y.min()) / 3000)
    width  = max(width, 2)
    height = max(height, 2)

    fig_main, ax_main = plt.subplots(figsize=(width, height))


    fig_main.patch.set_alpha(0.0)
    ax_main.set_facecolor('none')

    im = ax_main.imshow(
        H_norm.T,
        origin='lower',
        extent=extent,
        cmap=cmap_obj,
        interpolation='nearest',
        vmin=0.0,
        vmax=1.0
    )

    ax_main.invert_yaxis()
    ax_main.set_xticks([])
    ax_main.set_yticks([])

    plt.tight_layout()

    if main_png is None:
        main_png = f'E:/CL/HD/PAPER_WRITE/Figure1/figures/gene_whole_brain/{marker}_density.png'
    fig_main.savefig(main_png, dpi=600, bbox_inches='tight', transparent=True)
    plt.close(fig_main)


    norm = Normalize(vmin=0.0, vmax=1.0)
    sm = ScalarMappable(norm=norm, cmap=cmap_obj)
    sm.set_array([])

    cm = 1 / 2.54
    fig_cbar, cax = plt.subplots(
        figsize=(0.8 * cm, 4.8 * cm)
    )

    fig_cbar.patch.set_alpha(0.0)
    cax.set_facecolor('none')

    cb = plt.colorbar(sm, cax=cax)
    cb.set_label(f'{marker} normalized density (0–1)')
    cb.set_ticks([0.0, 0.5, 1.0])
    cb.set_ticklabels(['0', '0.5', '1'])

    plt.tight_layout()

    if cbar_png is None:
        cbar_png = f'E:/CL/HD/PAPER_WRITE/Figure1/figures/gene_whole_brain/{marker}_density_cbar.png'
    fig_cbar.savefig(cbar_png, dpi=600, bbox_inches='tight', transparent=True)
    plt.close(fig_cbar)

    print(f'{marker}: main plot -> {main_png} (normalized to 0-1)')
    print(f'{marker}: colorbar -> {cbar_png} (normalized to 0-1)')
    print(f'{marker}: density limits dmin={dmin}, dmax={dmax}')


In [ ]:
markers = [
    'Mbp', 'Ccdc153', 'Sst', 'Gja1',
    'Pvalb', 'Vip',
    'Slc17a7', 'Slc17a6', 'Gad1', 'Slc1a3', 'Lamp5',
    'Drd1', 'Drd2', 'Calb2', 'Cldn11', 'Vtn', 'Cpne4',
]

density_ranges = {
    'Mbp':     (1, 200),
    'Ccdc153': (2, 30),
    'Sst':     (1, 25),
    'Gja1':    (1, 10),
    'Pvalb':   (1, 20),
    'Vip':     (1, 20),
    'Slc17a7': (1, 20),
    'Slc17a6': (1, 20),
    'Gad1':    (1, 20),
    'Slc1a3':  (1, 20),
    'Lamp5':   (1, 20),
    'Drd1':    (1, 20),
    'Drd2':    (1, 20),
    'Calb2':   (1, 20),
    'Cldn11':  (1, 20),
    'Vtn':     (1, 20),
    'Cpne4':   (1, 20),
}


In [ ]:
for mk in markers:
    dmin, dmax = density_ranges[mk]
    draw_density_with_separate_cbar(
        mk,
        dmin=dmin,
        dmax=dmax,
        cmap='viridis',
    )


## Panel H | Puncta and segmentation overlay

Requires nuclear labels, expanded labels, and a spot CSV with `x`, `y`, and `brightness` columns. This routine draws segmentation boundaries and spots; raw mHTT/DAPI image display is not included.


In [ ]:
# Imports and settings.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tifffile import imread


def _binary_dilate(mask: np.ndarray, iters: int = 1) -> np.ndarray:
    """Dilate a binary mask with a 3-by-3 neighborhood."""
    m = mask.astype(bool)
    for _ in range(max(1, int(iters))):
        p = np.pad(m, 1, mode="constant", constant_values=False)
        m = (
            p[:-2, :-2] | p[:-2, 1:-1] | p[:-2, 2:] |
            p[1:-1, :-2] | p[1:-1, 1:-1] | p[1:-1, 2:] |
            p[2:, :-2] | p[2:, 1:-1] | p[2:, 2:]
        )
    return m


def _label_boundary_mask(lbl: np.ndarray) -> np.ndarray:
    """Find nonzero label boundaries using four neighbors."""


    lbl = lbl.astype(np.int32)
    p = np.pad(lbl, 1, mode="constant", constant_values=0)
    c = p[1:-1, 1:-1]
    up = p[:-2, 1:-1]
    dn = p[2:, 1:-1]
    lf = p[1:-1, :-2]
    rt = p[1:-1, 2:]
    return (c != 0) & ((c != up) | (c != dn) | (c != lf) | (c != rt))


def _binary_boundary_mask(mask: np.ndarray) -> np.ndarray:
    """Find binary mask boundaries using four neighbors."""
    m = mask.astype(bool)
    p = np.pad(m, 1, mode="constant", constant_values=False)
    c = p[1:-1, 1:-1]
    up = p[:-2, 1:-1]
    dn = p[2:, 1:-1]
    lf = p[1:-1, :-2]
    rt = p[1:-1, 2:]
    return c & (~up | ~dn | ~lf | ~rt)


def make_nuc_dilate_scatter_overlay(
    nuc_label_tif: str,
    dilate_label_tif: str,
    csv_path: str,
    out_png: str,
    x_col: str = "x",
    y_col: str = "y",
    val_col: str = "intensity",

    bg: str = "black",
    ring_gray: float = 0.55,
    draw_cyto_boundary: bool = True,
    boundary_color=(1.0, 0.0, 0.0),
    boundary_thickness: int = 2,

    point_size: float = 2.0,
    point_alpha: float = 1.0,
    cmap: str = "magma",
    vmin=None, vmax=None,
    clip_percentiles=(1, 99),
    flip_y: bool = False,
    dpi: int = 600,
    add_colorbar: bool = True,
):

    nuc = imread(nuc_label_tif)
    dil = imread(dilate_label_tif)

    if nuc.shape != dil.shape:
        raise ValueError(f"Shape mismatch: nuc {nuc.shape} vs dil {dil.shape}")

    H, W = nuc.shape[:2]
    nuc_mask = nuc > 0
    dil_mask = dil > 0
    ring_mask = dil_mask & (~nuc_mask)


    if bg == "black":
        base = np.zeros((H, W, 3), dtype=np.float32)
        transparent = False
    elif bg == "white":
        base = np.ones((H, W, 3), dtype=np.float32)
        transparent = False
    elif bg == "none":
        base = np.zeros((H, W, 4), dtype=np.float32)
        transparent = True
    else:
        raise ValueError("bg must be 'black', 'white', or 'none'")

    def paint(mask, rgb, alpha=1.0):
        if base.shape[2] == 3:
            base[mask, 0] = rgb[0]
            base[mask, 1] = rgb[1]
            base[mask, 2] = rgb[2]
        else:
            base[mask, 0] = rgb[0]
            base[mask, 1] = rgb[1]
            base[mask, 2] = rgb[2]
            base[mask, 3] = alpha


    paint(ring_mask, (ring_gray, ring_gray, ring_gray), alpha=1.0)
    paint(nuc_mask,  (1.0, 1.0, 1.0), alpha=1.0)


    if draw_cyto_boundary:

        unique_vals = np.unique(dil)
        is_label = (unique_vals.size > 2) or (dil.max() > 1)

        if is_label:
            bnd = _label_boundary_mask(dil)
        else:
            bnd = _binary_boundary_mask(dil_mask)

        if boundary_thickness > 1:
            bnd = _binary_dilate(bnd, iters=boundary_thickness - 1)

        paint(bnd, boundary_color, alpha=1.0)


    df = pd.read_csv(csv_path)
    x = df[x_col].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)
    v = df[val_col].to_numpy(dtype=float)

    if flip_y:
        y = (H - 1) - y

    if vmin is None or vmax is None:
        lo, hi = np.percentile(v, clip_percentiles)
        if vmin is None: vmin = lo
        if vmax is None: vmax = hi
    if vmax <= vmin:
        vmax = vmin + 1e-9


    fig_w_in = W / 300
    fig_h_in = H / 300
    fig, ax = plt.subplots(figsize=(fig_w_in, fig_h_in))

    ax.imshow(base[..., :3], origin="upper")
    sc = ax.scatter(
        x, y, c=v, s=point_size, cmap=cmap,
        vmin=0, vmax=1000, alpha=point_alpha,
        linewidths=0
    )

    ax.set_xlim(0, W - 1)
    ax.set_ylim(H - 1, 0)
    ax.set_axis_off()

    if add_colorbar:
        cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
        cbar.outline.set_linewidth(0)

    fig.savefig(out_png, dpi=dpi, bbox_inches="tight", pad_inches=0, transparent=transparent)
    plt.close(fig)
    print("Saved:", out_png, "| vmin/vmax:", vmin, vmax)


In [ ]:
make_nuc_dilate_scatter_overlay(
    nuc_label_tif = r"Label Image.tif",
    dilate_label_tif = r"LabelImage-Dilation.tif",
    csv_path = r"spot.csv",
    out_png = r"overlay.png",
    x_col="x", y_col="y", val_col="brightness",
    bg="black",
    cmap="viridis",
    point_size=3.0,
    clip_percentiles=(1, 99),
    flip_y=False
)
